### load libraries

In [ ]:
import yaml
from mstr_robotics._paths import REPO_ROOT, CONFIG_DIR, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
import pandas as pd
import uuid
import json
from mstr_robotics import regam
from mstr_robotics._helper import Misc
from mstr_robotics.mstr_classes import get_conn,MdSearches
from mstr_robotics.report import Prompts

run_id = uuid.uuid1().__str__()
i_test=regam.TestExe()
i_msic=Misc()
i_prompts=Prompts()
i_get_conn=get_conn
i_regam=regam.Regam({"run_id": run_id})
i_md_search=MdSearches()
i_regam_jobs=regam.RegamJobs()

with open(USER_CONFIG, 'r') as openfile:
    user_d = yaml.safe_load(openfile)
conn_params = user_d["conn_params"]

with open(CONFIG_DIR / "jupyter_objects_d.yml", "r") as openfile:
    jupyter_objects_d = yaml.safe_load(openfile)
nb_d = jupyter_objects_d["jup_REGAM"]


#set user credentials and open a connection to the i-server


### Config

In [ ]:
#get the jobs stored in a static report for this use case.
#object ids are maintained in ..\config\jupyter_objects_d.yml

conn=get_conn(**conn_params)

pa_project_id = user_d["mstr_projects"]["pa_project_id"]
pa_base_url = conn_params["base_url"]
pa_report_id=nb_d["reports"]["pa_report_id"]
pa_conn=get_conn(**conn_params)
pa_conn.select_project(pa_project_id)

### Prepare Testexecution

In [ ]:
# Select jobs

pa_raw_data_df=i_regam.fetch_pa_rep_jobs(pa_conn=pa_conn, pa_report_id=pa_report_id)
selected_rows = i_regam.select_rows_by_job_id(pa_raw_data_df)

# prepare execution JSON
all_jobs_prp_ans_JSON_d=i_regam.run_bld_job_prp_JSON(conn,pa_raw_data_df=selected_rows)
all_jobs_prp_ans_JSON_d

### Execute test

In [ ]:
# create testReports
i_test.run_test_exe(conn=conn,all_jobs_prp_ans_JSON_l=all_jobs_prp_ans_JSON_d["all_jobs_prp_ans_JSON_l"])